In [ ]:
#importing packages and env-var

from pyspark.sql.session import SparkSession
import pyspark.sql.functions as f

import os
from dotenv import load_dotenv

env = load_dotenv('/mnt/d/linux/project/vars/.env')

In [ ]:
#load env-vars
pgconn = os.getenv('POSTGRE_CONN')
pgdriver = os.getenv('PG_DRIVER')
pgjar = os.getenv('POSTGRE_JAR')
pguser = os.getenv('POSTGRE_USER')
pgpwd = os.getenv('POSTGRE_PWD')

In [ ]:
#create session 
spark = (( 
          SparkSession
          .builder
          .appName('invoices')
          .config('spark.jars', f'{pgjar}')
          .config('spark.executor.extraClassPath', f'{pgjar}')
          .config('spark.driver.extraClassPath', f'{pgjar}')
          .getOrCreate()
        ))

In [12]:
#import invoice data from sales.invoice table

tmp_invoiceline_frame = ((spark
                            .read
                            .format('jdbc')
                            .option('url', pgconn)
                            .option('driver', pgdriver)
                            .option('password', pgpwd)
                            .option('user', pguser)
                            .option('query', 
                                        
                                        """ 
                                            select
                                            
                                                invoiceid,
                                                invoicelineid,
                                                stockitemid,
                                                packagetypeid,
                                                quantity,
                                                unitprice,
                                                taxamount,
                                                extendedprice,
                                                lineprofit,
                                                taxrate,
                                                _airbyte_ab_id,
                                                _airbyte_invoicelines_hashid as invoiceline_hashid,
                                                _airbyte_emitted_at as elt_start,
                                                _airbyte_normalized_at as elt_end
                                                
                                            from sales.invoicelines

                                        """
                                    )
                        )).load().withColumnRenamed('_airbyte_ab_id', 'elt_invoiceline_hashid')

In [11]:
tmp_invoiceorder_frame = ((spark
                           .read
                           .format('jdbc')
                           .option('driver', pgdriver)
                           .option('url', pgconn)
                           .option('user', pguser)
                           .option('password', pgpwd)
                           .option('query', 
                                    """ 
                                    
                                        select
                                        
                                            orderid,
                                            invoiceid,
                                            customerid,
                                            salespersonpersonid,
                                            accountspersonid,
                                            customerpurchaseordernumber,
                                            deliverymethodid,
                                            billtocustomerid,
                                            contactpersonid,
                                            packedbypersonid,
                                            invoicedate,
                                            cast(confirmeddeliverytime as date) as deliverydate,
                                            totaldryitems,
                                            totalchilleritems,
                                            returneddeliverydata,
                                            _airbyte_ab_id,
                                            _airbyte_invoices_hashid as invoices_hashid,
                                            _airbyte_emitted_at as elt_start,
                                            _airbyte_normalized_at as elt_end
                                            
                                        from sales.invoices
                                    
                                    """
                                   )
                        )).load().withColumnRenamed('_airbyte_ab_id', 'elt_invoices_hahsid')

In [16]:
#joining dataframes 

join_invoiceframes = (( tmp_invoiceline_frame
                        .join(tmp_invoiceorder_frame, 'invoiceid', 'left')
                            
                        ))

In [17]:
join_invoiceframes.show()

23/11/01 12:43:23 ERROR Executor: Exception in task 0.0 in stage 4.0 (TID 4)
org.postgresql.util.PSQLException: Connection to localhost:5435 refused. Check that the hostname and port are correct and that the postmaster is accepting TCP/IP connections.
	at org.postgresql.core.v3.ConnectionFactoryImpl.openConnectionImpl(ConnectionFactoryImpl.java:342)
	at org.postgresql.core.ConnectionFactory.openConnection(ConnectionFactory.java:54)
	at org.postgresql.jdbc.PgConnection.<init>(PgConnection.java:263)
	at org.postgresql.Driver.makeConnection(Driver.java:443)
	at org.postgresql.Driver.connect(Driver.java:297)
	at org.apache.spark.sql.execution.datasources.jdbc.connection.BasicConnectionProvider.getConnection(BasicConnectionProvider.scala:49)
	at org.apache.spark.sql.execution.datasources.jdbc.connection.ConnectionProviderBase.create(ConnectionProvider.scala:102)
	at org.apache.spark.sql.jdbc.JdbcDialect.$anonfun$createConnectionFactory$1(JdbcDialects.scala:160)
	at org.apache.spark.sql.jdbc

Py4JJavaError: An error occurred while calling o100.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 5.0 failed 1 times, most recent failure: Lost task 0.0 in stage 5.0 (TID 5) (172.24.129.215 executor driver): org.postgresql.util.PSQLException: Connection to localhost:5435 refused. Check that the hostname and port are correct and that the postmaster is accepting TCP/IP connections.
	at org.postgresql.core.v3.ConnectionFactoryImpl.openConnectionImpl(ConnectionFactoryImpl.java:342)
	at org.postgresql.core.ConnectionFactory.openConnection(ConnectionFactory.java:54)
	at org.postgresql.jdbc.PgConnection.<init>(PgConnection.java:263)
	at org.postgresql.Driver.makeConnection(Driver.java:443)
	at org.postgresql.Driver.connect(Driver.java:297)
	at org.apache.spark.sql.execution.datasources.jdbc.connection.BasicConnectionProvider.getConnection(BasicConnectionProvider.scala:49)
	at org.apache.spark.sql.execution.datasources.jdbc.connection.Connecti